# RAG Pipeline Improvement — Results

This notebook is the single, readable place for the experiment results. It reads
`experiments/results/runs.csv` (written by `harness.run_arm`) and renders the
story in run order: **Stage A (chunking) → B (retrieval) → C (embedding) → K (k selection)**,
then worked examples and the LLM-judge scores.

Every number is data-derived — re-run the cells after each stage to refresh.
The GitHub/PR-friendly twin is `IMPROVEMENT_REPORT.md` (auto-generated by `report.py`).

> Read first: `experiments/v2/README.md` for how to run the stages.

In [ ]:
import sys, csv
from pathlib import Path
import pandas as pd

ROOT = Path.cwd()
while ROOT != ROOT.parent and not (ROOT / 'experiments' / 'results').exists():
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))
from experiments.v2.report import _aggregate, _winner, _stage_of, K_VALUES, _TABLE_METRICS

RUNS_CSV = ROOT / 'experiments' / 'results' / 'runs.csv'
runs = list(csv.DictReader(open(RUNS_CSV, encoding='utf-8'))) if RUNS_CSV.exists() else []
for r in runs:
    for c in ('score_norm','raw_score','pool_k','rerank_k','context_k'):
        try: r[c] = float(r[c])
        except (TypeError, ValueError): pass
agg = _aggregate(runs) if runs else {}
print(f'{len(runs)} run rows across {len(agg)} arms')

## Headline

Best arm overall, the recommended pipeline, and the key deltas vs the baseline.

In [ ]:
if agg:
    best = _winner(agg)
    m = agg[best]
    print(f'Best arm: {best}')
    print(f"  chunking={m.get('chunking')}  embedding={m.get('embedding')}  retrieval={m.get('retrieval')}")
    print(f"  R@5={m.get('recall@5',0):.3f}  MRR={m.get('mrr',0):.3f}  p95={m.get('latency_p95_ms',0):.0f}ms")
else:
    print('No runs yet — execute run_stage.py first.')

In [ ]:
def stage_table(stage_letter):
    arms = {a: m for a, m in agg.items() if _stage_of(a) == stage_letter}
    if not arms:
        print(f'Stage {stage_letter}: no runs yet'); return None
    df = pd.DataFrame(arms).T[[c for c in _TABLE_METRICS if any(c in v for v in arms.values())]]
    df.loc['WINNER'] = ''
    win = _winner(arms)
    print(f'Stage {stage_letter} winner: {win}')
    return df.drop(index='WINNER')

## Stage A — Chunking (embedding + retrieval fixed)

Variable changed: chunking only. Explain *why* the winner won (e.g. breadcrumb
context helped split chunks keep their specialization).

In [ ]:
stage_table('A')

## Stage B — Retrieval (chunking = winner A, embedding fixed)

Variable changed: retrieval method only. Note the marginal value and latency of
each added component (BM25/sparse → RRF → rerank → query expansion).

In [ ]:
stage_table('B')

## Stage C — Embedding (chunking + retrieval locked)

Variable changed: embedder only. Does the Hebrew-specialized NeoDictaBERT beat
gemini-embedding-2 on this corpus? Weigh quality vs cost/latency/index size.

In [ ]:
stage_table('C')

## Stage K — k selection (recall vs latency knee)

The chart below (`stage_k_knee.png`, written by `select_k.py`) shows Recall/nDCG
and p95 latency vs `context_k`. The recommended `(pool_k, rerank_k, context_k)`
is the knee within the latency budget.

In [ ]:
from IPython.display import Image, display
knee = ROOT / 'experiments' / 'results' / 'stage_k_knee.png'
display(Image(str(knee))) if knee.exists() else print('Run select_k.py to produce the knee chart.')

## Worked examples (before vs after)

Pick 2-3 real Hebrew questions and show the retrieved chunks for the baseline
arm vs the winning arm, so the metric gains are tangible. Fill in `QUESTIONS`
and run against the locked pipeline via `harness.build_context` + `run_retrieval`.

In [ ]:
# Example (uncomment after stages run; performs real retrieval):
# from experiments.v2 import chunkers, harness, retrievers
# from experiments.v2.configs import ArmConfig
# QUESTIONS = ['כמה נקודות זכות ההתמחות בפינטך?']
# cfg = ArmConfig(arm_id='demo', chunking='breadcrumb', embedding='gemini-embedding-2', retrieval='rerank_ce')
# chunks = chunkers.build_chunks(cfg.chunking)
# ctx = harness.build_context(cfg, chunks)
# for q in QUESTIONS:
#     hits = retrievers.run_retrieval(cfg.retrieval, q, ctx, cfg.pool_k, cfg.rerank_k, cfg.context_k)
#     print(q); [print(' -', h['chunk_id'], h['text'][:80]) for h in hits]
print('Fill in QUESTIONS and uncomment to render worked examples.')

## LLM-judge scores (Stage P)

After promotion, the 5 judges (faithfulness, correctness, context_relevance,
answer_relevance, completeness) score the live pipeline. Load that results table
(`evaluation_framework.runner.run` output) here and summarize per metric.

In [ ]:
judge_csv = ROOT / 'experiments' / 'results' / 'judge_scores.csv'
if judge_csv.exists():
    jdf = pd.read_csv(judge_csv)
    display(jdf.groupby('metric')['score_norm'].mean().round(3))
else:
    print('Run the Stage P full evaluation to produce judge_scores.csv.')